In [ ]:
#from google.colab import files
#uploaded = files.upload()  # a file picker will appear

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.5 MB/s eta 0:00:00


roboflow [project](https://app.roboflow.com/toshans-workspace-eto4j/home)

In [ ]:
!pip install roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="REPLACE")
project = rf.workspace().project("badminton-shuttlecock-dv7zr")
model = project.version(3).model

loading Roboflow workspace...
loading Roboflow project...


This code creates boxes around the players and their positions
- working

In [ ]:
# from ultralytics import YOLO
# import cv2

# model = YOLO('yolov8n.pt')

# cap = cv2.VideoCapture('Badminton_Rally.mp4')
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter('output.mp4', fourcc, 30.0,
#                          (int(cap.get(3)), int(cap.get(4))))

# while cap.isOpened():
#   ret, frame = cap.read()
#   if not ret:
#     break

#   results = model(frame, classes=[0]) #class[0] = for people
#   annotated = results[0].plot()
#   out.write(annotated)

# cap.release()
# out.release()
# print("Done!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

0: 384x640 2 persons, 385.7ms
Speed: 17.8ms preprocess, 385.7ms inference, 50.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 155.3ms
Speed: 5.9ms preprocess, 155.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 154.9ms
Speed: 6.7ms preprocess, 154.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 151.6ms
Speed: 4.6ms preprocess, 151.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 160.9ms
Speed: 3.8ms preprocess, 160.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 237.3ms
Speed: 6.8m

this intended to add skeletons of the joints of the players
- edit: added shuttle detection too

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
import cv2

# Load pose model
pose_model = YOLO('yolov8n-pose.pt')

# Load shuttlecock model
rf = Roboflow(api_key="sjPEQQVcQlyVjQfZT5Qa")
project = rf.workspace().project("badminton-shuttlecock-dv7zr")
shuttle_model = project.version(3).model

cap = cv2.VideoCapture('Badminton_Rally.mp4')
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_final.mp4', fourcc,
      cap.get(cv2.CAP_PROP_FPS),
      (int(cap.get(3)), int(cap.get(4))))

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run pose detection
    pose_results = pose_model(frame, verbose=False)
    annotated = pose_results[0].plot()

    # Run shuttlecock detection every 3 frames (saves time)
    if frame_count % 3 == 0:
        cv2.imwrite('temp_frame.jpg', frame)
        predictions = shuttle_model.predict('temp_frame.jpg', confidence=30).json()['predictions']

    # Draw shuttlecock dot
    for pred in predictions:
        cx = int(pred['x'])
        cy = int(pred['y'])
        cv2.circle(annotated, (cx, cy), 12, (0, 255, 255), -1)  # yellow dot
        cv2.putText(annotated, f"{int(pred['confidence']*100)}%",
                    (cx+15, cy), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 255, 255), 2)

    out.write(annotated)
    frame_count += 1

    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print("Done!")

loading Roboflow workspace...
loading Roboflow project...
Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processed 390 frames...
Processed 420 frames...
Processed 450 frames...
Processed 480 frames...
Processed 510 frames...
Processed 540 frames...
Processed 570 frames...
Done!


In [ ]:
from google.colab import files
files.download('output_final.mp4')    #for checking after

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ts just debugging to see if it read the file, could be useful later


In [ ]:
# import cv2

# cap = cv2.VideoCapture('your_video.mp4')

# print("Video opened:", cap.isOpened())
# print("Frame width:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# print("Frame height:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# print("FPS:", cap.get(cv2.CAP_PROP_FPS))
# print("Total frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))

# ret, frame = cap.read()
# print("First frame read:", ret)
# cap.release()

Video opened: False
Frame width: 0.0
Frame height: 0.0
FPS: 0.0
Total frames: 0.0
First frame read: False


test for one frame to see if shuttle detection is working

In [ ]:
# import cv2

# cap = cv2.VideoCapture('Badminton_Rally.mp4')
# ret, frame = cap.read()
# cap.release()

# cv2.imwrite('test_frame.jpg', frame)

# # Run shuttlecock detection on it
# prediction = model.predict('test_frame.jpg', confidence=30).json()
# print(prediction)

{'predictions': [{'x': 1306.5, 'y': 886.0, 'width': 31.0, 'height': 30.0, 'confidence': 0.8818392753601074, 'class': 'Shuttlecock', 'class_id': 0, 'detection_id': 'd67854b6-6012-4507-bb91-74d320551809', 'image_path': 'test_frame.jpg', 'prediction_type': 'ObjectDetectionModel'}], 'image': {'width': '1948', 'height': '1104'}}


using youtube video

In [ ]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.4 MB/s eta 0:00:00


axelson game,
momota game

In [ ]:
!yt-dlp -f 'best[ext=mp4]' "https://www.youtube.com/watch?v=gmXVbt-AC8o" -o "Bellevue_game.mp4"

[youtube] Extracting URL: https://www.youtube.com/watch?v=gmXVbt-AC8o
[youtube] gmXVbt-AC8o: Downloading webpage
[youtube] gmXVbt-AC8o: Downloading android vr player API JSON
[info] gmXVbt-AC8o: Downloading 1 format(s): 18
[download] Destination: Bellevue_game.mp4
[download] 100% of   29.14MiB in 00:00:02 at 11.30MiB/s


In [ ]:
#!yt-dlp -f 'best[ext=mp4]' "https://www.youtube.com/watch?v=gmXVbt-AC8o" -o "Bellevue_game.mp4"
!yt-dlp -f 'best[ext=mp4]' --download-sections "*0:00-5:00" "https://www.youtube.com/watch?v=UZ2mynjp0yY" -o "momota.mp4"

[youtube] Extracting URL: https://www.youtube.com/watch?v=UZ2mynjp0yY
[youtube] UZ2mynjp0yY: Downloading webpage
[youtube] UZ2mynjp0yY: Downloading android vr player API JSON
[info] UZ2mynjp0yY: Downloading 1 format(s): 18
[info] UZ2mynjp0yY: Downloading 1 time ranges: 0.0-300.0
[download] Destination: momota.mp4
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from 'https://rr5---sn-2onx5c-5p.googlevideo.com/videoplayback?expire=1781995273&ei=qcI2aoTVF733sfIPwqPYwQI&ip=35.252.133.240&id=o-AFNb_sYVuFwg4H_qYFkqqn0QJ59Ker09_l7p8lQ3eoCa&itag=18&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&cps=307&met=1781973673%2C&mh=ca&mm=31%2C26&mn=sn-2onx5c-5p%2Csn-a5meknds&ms=au%2Conr&mv=m&mvi=5&pl=17&rms=au%2Cau&bui=ARmQxEVBbYx22wK4FdvwLoDKza7d5U7seKOKDn4a3L7rXjzi0vVGuIblG6fhIrMvro-kaVjRA1384Lwc&spc=SQ-umrfCu7tKYZ6j7coqYPm5eJmMvyG35DNreEjTohyrjPE6XkrH&vprv=1&svpuc=1&mime=video%2Fmp4&rqh=1&cnr=14&ratebypass=yes&dur=423.067&lmt=1756974196913967&mt=1781973187&fvip=5&fexp=51565115%2C51565682%2C51987687&c=ANDROID


next three just to see if it downloaded + video stats

In [ ]:
import os
print(os.listdir())

['.config', 'drive', 'sample_data']


In [ ]:
import os
size = os.path.getsize('Badminton_Rally.mp4') / (1024 * 1024)
print(f"File size: {size:.1f} MB")

FileNotFoundError: [Errno 2] No such file or directory: 'Badminton_Rally.mp4'

In [ ]:
import cv2
cap = cv2.VideoCapture('Bellevue_game.mp4')
print("Opened:", cap.isOpened())
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Duration:", cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS), "seconds")
cap.release()

Opened: True
FPS: 29.97
Duration: 1008.2082082082082 seconds


now save preview frames every 3 seconds to find contact frames

In [ ]:
import cv2
import os

os.makedirs('frames_preview', exist_ok=True)
cap = cv2.VideoCapture('Bellevue_game.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
interval = int(fps * 3)  # every 3 seconds
frame_count = 0
saved = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % interval == 0:
        timestamp = frame_count / fps
        filename = f'frames_preview/frame_{timestamp:.1f}s.jpg'
        cv2.imwrite(filename, frame)
        saved += 1
    frame_count += 1

cap.release()
print(f"Saved {saved} preview frames")

Saved 340 preview frames


better version of up code, run for every 2 min of video

In [ ]:
import cv2
import os

os.makedirs('frames_preview2', exist_ok=True)
cap = cv2.VideoCapture('Bellevue_game.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)

# Just look at first 2 minutes to start
start_sec = 0
end_sec = 120

interval = int(fps * 0.5)  # every 0.5 seconds
frame_count = 0
saved = 0

cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_sec * fps))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    current_sec = frame_count / fps + start_sec
    if current_sec > end_sec:
        break

    if frame_count % interval == 0:
        timestamp = current_sec
        filename = f'frames_preview2/frame_{timestamp:07.2f}s.jpg'
        cv2.imwrite(filename, frame)
        saved += 1

    frame_count += 1

cap.release()
print(f"Saved {saved} preview frames")

Saved 257 preview frames


download folder of preview frames

In [ ]:
import shutil
shutil.make_archive('frames_preview2', 'zip', 'frames_preview2')

from google.colab import files
files.download('frames_preview2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

hopefully better way of seeing frames
- update, its way better
- update, its torture


In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

cap = cv2.VideoCapture('momota.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_mins = total_frames / fps / 60

coarse = widgets.FloatSlider(
    value=0, min=0, max=duration_mins,
    step=1, description='Minute:',
    layout=widgets.Layout(width='900px')
)

fine = widgets.IntSlider(
    value=0, min=0, max=int(fps*60)-1,
    step=1, description='Frame:',
    layout=widgets.Layout(width='900px')
)

output = widgets.Output()

def update(change):
    frame_num = int(coarse.value * 60 * fps) + fine.value
    frame_num = min(frame_num, total_frames - 1)

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if not ret:
        return

    frame = cv2.resize(frame, (960, 540))
    timestamp = frame_num / fps
    cv2.putText(frame, f"Frame: {frame_num} | Time: {timestamp:.2f}s",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    _, buffer = cv2.imencode('.jpg', frame)
    with output:
        clear_output(wait=True)
        display(Image(data=buffer.tobytes()))

coarse.observe(update, names='value')
fine.observe(update, names='value')

display(coarse, fine, output)
update(None)

FloatSlider(value=0.0, description='Minute:', layout=Layout(width='900px'), max=5.000556111667223, step=1.0)

IntSlider(value=0, description='Frame:', layout=Layout(width='900px'), max=1797)

Output()

putting data into arrays names labeled_moments_blahblah

In [ ]:
labeled_moments_axelsen = [
    (50.78, 'backhand_serve'),
    (52.59, 'clear'),
    (55.02, 'net'),
    (57.29, 'smash'),
    (58.63, 'net'),
    (59.66, 'backhand'),
    (82.35, 'backhand_serve'),
    (84.22, 'net'),
    (100.03, 'net'),
    (111.51, 'backhand_serve'),
    (113.45, 'net'),
    (115.62, 'smash'),
    (117.22, 'net'),
    (119.25, 'net'),
    (121.19, 'net'),
    (122.76, 'backhand'),
    (144.24, 'backhand_serve'),
    (145.85, 'net'),
    (148.01, 'net'),
    (150.18, 'drop'),
    (151.85, 'net'),
    (167.03, 'backhand_serve'),
    (168.54, 'backhand'),
    (170.04, 'net'),
    (172.57, 'smash'),
    (174.04, 'net'),
    (175.98, 'net'),
    (176.98, 'net'),
    (178.48, 'net'),
    (203.10, 'net'),
    (205.31, 'smash'),
    (206.67, 'net'),
    (209.14, 'smash'),
    (224.82, 'backhand_serve'),
    (227.36, 'smash'),
    (229.23, 'net'),
    (231.13, 'net'),
    (253.62, 'smash'),
    (292.26, 'smash'),
    (296.46, 'backhand'),
    (326.29, 'backhand_serve'),
    (328.13, 'drop'),
    (365.80, 'backhand_serve'),
    (378.75, 'drop'),
    (388.22, 'smash'),
    (394.36, 'smash'),
    (420.85, 'backhand_serve'),
    (462.06, 'backhand'),
    (463.60, 'net'),
    (465.50, 'net'),
    (483.55, 'drop'),
    (487.22, 'drop'),
    (512.95, 'backhand_serve'),
    (514.85, 'backhand'),
    (516.85, 'smash'),
    (530.93, 'smash'),
    (532.63, 'net'),
    (535.04, 'drop'),
    (539.11, 'clear'),
    (552.09, 'backhand_serve'),
    (561.86, 'smash'),
    (579.75, 'backhand_serve'),
]

labeled_moments_momota = [
    (3.07, 'smash'),
    (4.70, 'backhand_serve'),
    (6.51, 'smash'),
    (9.04, 'net'),
    (12.75, 'net'),
    (15.05, 'clear'),
    (17.15, 'drop'),
    (24.72, 'backhand'),
    (28.93, 'backhand'),
    (32.97, 'clear'),
    (36.80, 'clear'),
    (39.21, 'smash'),
    (42.51, 'backhand'),
    (48.65, 'smash'),
    (52.39, 'drop'),
    (57.96, 'backhand_serve'),
    (60.03, 'backhand'),
    (64.56, 'backhand'),
    (66.97, 'drop'),
    (77.91, 'clear'),
    (85.55, 'drop'),
    (92.23, 'backhand'),
    (115.22, 'backhand'),
    (116.75, 'backhand'),
    (121.05, 'clear'),
    (126.66, 'smash'),
    (130.33, 'backhand'),
    (149.78, 'drop'),
    (153.69, 'clear'),
    (160.93, 'smash'),
    (164.50, 'backhand'),
    (168.34, 'drop'),
    (173.67, 'backhand'),
    (177.78, 'clear'),
    (187.99, 'clear'),
    (190.09, 'smash'),
    (196.80, 'drop'),
    (201.30, 'smash'),
    (220.45, 'drop'),
    (239.27, 'clear'),
    (246.51, 'drop'),
    (286.35, 'smash'),
    (289.79, 'drop'),
]

extract keypoints from both and put into a CSV

In [ ]:
from ultralytics import YOLO
import cv2
import csv

pose_model = YOLO('yolov8n-pose.pt')

def extract_keypoints(video_path, labeled_moments):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    rows = []

    for (timestamp, label) in labeled_moments:
        frame_number = int(timestamp * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()

        if not ret:
            print(f"Couldn't read frame at {timestamp}s")
            continue

        results = pose_model(frame, verbose=False)

        if results[0].keypoints is None or len(results[0].keypoints.xy) == 0:
            print(f"No person detected at {timestamp}s, skipping")
            continue

        kps = results[0].keypoints.xy[0].cpu().numpy()
        flat = kps.flatten().tolist()
        flat.append(label)
        rows.append(flat)
        print(f"Captured {label} at {timestamp}s")

    cap.release()
    return rows

# Extract from both videos
print("Processing Axelsen...")
rows_axelsen = extract_keypoints('Bellevue_game.mp4', labeled_moments_axelsen)

print("\nProcessing Momota...")
rows_momota = extract_keypoints('momota.mp4', labeled_moments_momota)

# Combine
all_rows = rows_axelsen + rows_momota

# Save to CSV
header = []
for i in range(17):
    header.append(f'kp{i}_x')
    header.append(f'kp{i}_y')
header.append('label')

with open('keypoints.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(all_rows)

print(f"\nDone! Saved {len(all_rows)} labeled frames to keypoints.csv")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Processing Axelsen...
Captured backhand_serve at 50.78s
Captured clear at 52.59s
Captured net at 55.02s
Captured smash at 57.29s
Captured net at 58.63s
Captured backhand at 59.66s
Captured backhand_serve at 82.35s
Captured net at 84.22s
Captured net at 100.03s
Captured backhand_serve at 111.51s
Captured net at 113.45s
Captured smash at 115.62s
Captured net at 117.22s
Captured net at 119.25s
Captured net at 121.19s
Captured backhand at 122.76s
Captured backhand_serve at 144.24s
Captured net at 145.85s
Captured net at 148.01s
Captured drop at 150.18s
No person detected at 151.85s, skipping
Captured backhand_serve at 167.03s
Captured backhand at 168.54s
Captured net at 170.04s
Captur

random forest classifier training



In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

# Load dataset
df = pd.read_csv('keypoints.csv')
X = df.drop('label', axis=1)
y = df['label']

print("Label distribution:")
print(y.value_counts())
print(f"\nTotal samples: {len(df)}")

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining on {len(X_train)} samples, testing on {len(X_test)} samples")

# Train
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
print("\n--- Results ---")
print(classification_report(y_test, model.predict(X_test)))

# Save
joblib.dump(model, 'shot_classifier.pkl')
print("Model saved to shot_classifier.pkl")

Label distribution:
label
net               24
smash             22
backhand          17
drop              16
backhand_serve    14
clear             11
Name: count, dtype: int64

Total samples: 104

Training on 83 samples, testing on 21 samples

--- Results ---
                precision    recall  f1-score   support

      backhand       1.00      0.67      0.80         3
backhand_serve       1.00      1.00      1.00         3
         clear       1.00      1.00      1.00         2
          drop       0.25      0.33      0.29         3
           net       0.80      0.80      0.80         5
         smash       0.80      0.80      0.80         5

      accuracy                           0.76        21
     macro avg       0.81      0.77      0.78        21
  weighted avg       0.80      0.76      0.77        21

Model saved to shot_classifier.pkl


better shot logic v1

*   Update, added is_player method to check both players
*   Update, wrist velocity checker to see when shots are played



In [ ]:
from ultralytics import YOLO
import cv2
import joblib
import numpy as np
import pandas as pd

pose_model = YOLO('yolov8n-pose.pt')
detect_model = YOLO('yolov8n.pt')
clf = joblib.load('shot_classifier.pkl')

def is_player(box, frame_w, frame_h):
    x1, y1, x2, y2 = box
    width = x2 - x1
    height = y2 - y1
    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2

    if height < width * 1.2:
        return False
    if center_y < frame_h * 0.2 or center_y > frame_h * 0.95:
        return False
    if center_x < frame_w * 0.1 or center_x > frame_w * 0.9:
        return False
    if height < frame_h * 0.15:
        return False
    return True

def get_court_zone(x, y, frame_width, frame_height):
    left = x < frame_width / 2
    back = y < frame_height / 2

    if left and back:
        return "back-left"
    elif not left and back:
        return "back-right"
    elif left and not back:
        return "front-left"
    else:
        return "front-right"

def opposite_zone(zone):
    opposites = {
        "back-left":   "front-right",
        "back-right":  "front-left",
        "front-left":  "back-right",
        "front-right": "back-left"
    }
    return opposites[zone]

def wrist_velocity(kps_flat, prev_pos):
    right_wrist_x = kps_flat[16*2]
    right_wrist_y = kps_flat[16*2 + 1]

    if prev_pos is None:
        return 0

    dx = right_wrist_x - prev_pos[0]
    dy = right_wrist_y - prev_pos[1]
    return np.sqrt(dx**2 + dy**2)

cap = cv2.VideoCapture('momota.mp4')
start_sec = 10
cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_sec * cap.get(cv2.CAP_PROP_FPS)))
max_frames = int(cap.get(cv2.CAP_PROP_FPS) * 30)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_analysis08.mp4', fourcc, fps, (frame_w, frame_h))

frame_count = 0
last_shot = None
last_suggestion = None
last_color = (0, 255, 0)
opp_zone = None
prev_wrist_pos = None
VELOCITY_THRESHOLD = 8
shot_display_frames = 0  # how many frames to keep label visible

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    annotated = frame.copy()

    # Detect all people as bounding boxes
    detect_results = detect_model(frame, classes=[0], verbose=False)
    all_boxes = detect_results[0].boxes.xyxy.cpu().numpy()
    boxes = [b for b in all_boxes if is_player(b, frame_w, frame_h)]

    if len(boxes) >= 2:
        areas = [(b[2]-b[0]) * (b[3]-b[1]) for b in boxes]
        sorted_boxes = [b for _, b in sorted(zip(areas, boxes), reverse=True)]
        near_box = sorted_boxes[0]
        far_box = sorted_boxes[1]

        opp_cx = int((far_box[0] + far_box[2]) / 2)
        opp_cy = int((far_box[1] + far_box[3]) / 2)
        opp_zone = get_court_zone(opp_cx, opp_cy, frame_w, frame_h)

        cv2.rectangle(annotated,
                     (int(far_box[0]), int(far_box[1])),
                     (int(far_box[2]), int(far_box[3])),
                     (0, 0, 255), 2)
        cv2.putText(annotated, f"Opponent: {opp_zone}",
                   (opp_cx - 60, int(far_box[1]) - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        cv2.rectangle(annotated,
                     (int(near_box[0]), int(near_box[1])),
                     (int(near_box[2]), int(near_box[3])),
                     (0, 255, 0), 2)

    # Run pose on full frame for shot classification
    pose_results = pose_model(frame, verbose=False)
    if pose_results[0].keypoints is not None and len(pose_results[0].keypoints.xy) > 0:
        kps = pose_results[0].keypoints.xy[0].cpu().numpy()
        kps_flat = kps.flatten()

        if kps_flat.max() > 0:
            velocity = wrist_velocity(kps_flat, prev_wrist_pos)
            prev_wrist_pos = (kps_flat[16*2], kps_flat[16*2 + 1])

            if velocity > VELOCITY_THRESHOLD:
                header = [f'kp{i}_{c}' for i in range(17) for c in ['x', 'y']]
                shot = clf.predict(pd.DataFrame([kps_flat], columns=header))[0]

                hip_x = kps_flat[23*2] if len(kps_flat) > 47 else frame_w / 2
                shot_zone = get_court_zone(hip_x, 0, frame_w, frame_h)

                if opp_zone is not None and len(boxes) >= 2:
                    better = opposite_zone(opp_zone)

                    if opp_zone == shot_zone:
                        suggestion = f"Should play to {better}!"
                        color = (0, 0, 255)
                    else:
                        suggestion = f"Good direction"
                        color = (0, 255, 0)

                    last_shot = shot
                    last_suggestion = suggestion
                    last_color = color
                    shot_display_frames = 45  # show label for 1.5 seconds

    # Only show shot label for a fixed number of frames after detection
    if shot_display_frames > 0:
        if last_shot:
            cv2.putText(annotated, f"Shot: {last_shot}",
                       (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 0), 3)
        if last_suggestion:
            cv2.putText(annotated, last_suggestion,
                       (30, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.0, last_color, 3)
        shot_display_frames -= 1

    out.write(annotated)
    frame_count += 1
    if frame_count >= max_frames:
        break

    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print("Done!")

Processed 30 frames...
Processed 60 frames...
Processed 90 frames...
Processed 120 frames...
Processed 150 frames...
Processed 180 frames...
Processed 210 frames...
Processed 240 frames...
Processed 270 frames...
Processed 300 frames...
Processed 330 frames...
Processed 360 frames...
Processed 390 frames...
Processed 420 frames...
Processed 450 frames...
Processed 480 frames...
Processed 510 frames...
Processed 540 frames...
Processed 570 frames...
Processed 600 frames...
Processed 630 frames...
Processed 660 frames...
Processed 690 frames...
Processed 720 frames...
Processed 750 frames...
Processed 780 frames...
Processed 810 frames...
Processed 840 frames...
Processed 870 frames...
Done!


trying NVIDIA LocateAnything

In [ ]:
!pip install transformers==4.57.1 accelerate

In [ ]:
!pip install decord lmdb

In [ ]:
from transformers import AutoTokenizer, AutoProcessor, AutoModel
import torch

model_id = "nvidia/LocateAnything-3B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).cuda().eval()

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


image_processing_locateanything.py: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Qwen2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
import cv2
from PIL import Image
import torch
import re

def get_shuttle_box(frame, prompt="Locate the badminton shuttlecock"):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

    inputs = processor(
        text=[text_prompt],
        images=[image],
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            use_cache=True,
            tokenizer=tokenizer
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True)
    return result

cap = cv2.VideoCapture('momota.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.set(cv2.CAP_PROP_POS_FRAMES, int(10 * fps))  # start at 10s
max_test_frames = 30  # only test on 30 frames first, this model is slow

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('shuttle_test.mp4', fourcc, fps, (frame_w, frame_h))

frame_count = 0

while cap.isOpened() and frame_count < max_test_frames:
    ret, frame = cap.read()
    if not ret:
        break

    annotated = frame.copy()

    # Only run every 2nd frame to save time during testing
    if frame_count % 2 == 0:
        raw_result = get_shuttle_box(frame)
        print(f"Frame {frame_count}: {raw_result}")

        # Try to extract coordinates - format depends on model output
        # This regex looks for patterns like [x1,y1,x2,y2] or similar
        numbers = re.findall(r'\d+\.?\d*', raw_result)

        if len(numbers) >= 4:
            x1, y1, x2, y2 = [int(float(n)) for n in numbers[:4]]
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(annotated, "shuttle", (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    out.write(annotated)
    frame_count += 1
    print(f"Processed {frame_count}/{max_test_frames}")

cap.release()
out.release()
print("Done!")

/root/.cache/huggingface/modules/transformers_modules/nvidia/LocateAnything_hyphen_3B/c32291ca5e996f5a7a485845b4f57a233936bba0/generate_utils.py:186: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  box_avg.append(torch.tensor(out_ref, dtype=x0.dtype, device=x0.device))


TypeError: argument 'ids': Can't extract `str` to `Vec`